In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
import scipy.io as sio
import tensorflow as tf
from scipy.stats import wasserstein_distance
import time

import models

In [2]:
eps = 0.01
T = 1

data = sio.loadmat("./data/test.mat")
ys = data["ys"]

In [3]:
model = models.NN(units=200, name="nn3n", activation=tf.tanh)
model.restore()
forward = tf.function(model.call)

In [11]:
M = 1000
idx = 2
yt = tf.constant(ys[idx, -1, :].reshape([1, -1]), tf.float32)
yt = tf.tile(yt, [M, 1])

In [12]:
dt = 0.01
zts = [yt]
t = 0
for i in range(int(np.ceil(T/dt))):
    update = eps * forward(
        (T - t) * tf.ones(shape=[M, 1]),
        tf.constant(zts[-1], tf.float32),
    ).numpy()
    new_zt = zts[-1] + update * dt + \
        np.sqrt(eps) * np.random.normal(size=zts[-1].shape) * np.sqrt(dt)
    zts += [new_zt]
    t = t + dt
    
zts = np.stack(zts, axis=0)
mus = np.mean(zts, axis=1)
sds = np.std(zts, axis=1)

In [13]:
x = np.linspace(0, 1, 100)

sio.savemat(
    "./outputs/result_{}.mat".format(str(idx)),
    {
        "x": x, "mus": mus, "sds": sds,
        "yt": ys[idx, -1, :],
        "y0": ys[idx, 0, :]
    }
)